In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
# Load your clean HEDIS code dictionary
ref_hedis_df = spark.read.table("claimspan.bronze.hedis_measures")

In [0]:
# Load your incoming raw EDI/Claims data feed 
# (Assumes standard EDI columns: member_id, claim_id, code, service_date, value)
raw_claims_df = spark.read.table("claimspan.bronze.raw_edi_claims")

In [0]:
# 2. INNER JOIN TO MAP CLINICAL EVENTS
# Filter claims to only those that match a tracked HEDIS code
mapped_claims_df = raw_claims_df.join(
    ref_hedis_df,
    on=(raw_claims_df.medical_code == ref_hedis_df.code),
    how="inner"
)

In [0]:
# 3. COMPUTE DERIVED FIELD LOGIC via SPARK EXPRESSIONS
silver_gaps_df = mapped_claims_df.select(
    # --- Autogenerated Keys & Core IDs ---
    F.md5(F.concat_ws("||", F.col("member_id"), F.col("measureCode"))).alias("memberMeasureID"),
    F.md5(F.concat_ws("||", F.col("member_id"), F.col("measureCode"), F.col("eventName"))).alias("memberServiceMeasureID"),
    F.hash(F.col("eventName")).alias("serviceMeasureID"),
    F.hash(F.col("measureCode")).alias("measureID"),
    F.lit(None).cast("integer").alias("subMeasureID"), # Can be refined per sub-cohort logic
    
    # --- Base Metadata ---
    F.col("measureCode"),
    F.col("measureName"),
    F.col("eventName"),
    
    # --- Category Mapping ---
    F.when(F.col("measureCode").isin("BPC-E", "CDC"), "Chronic Care")
     .when(F.col("measureCode").isin("BCS-E", "ASF-E"), "Preventive Screening")
     .otherwise("Behavioral Health").alias("category"),
     
    # --- Denominator & Numerator Logic ---
    F.lit(1).alias("denomCnt"), # If they match the measure criteria, they are in denominator
    
    # Example Clinical Calculation: Gap closes if code matches or specific thresholds clear
    F.when((F.col("eventName") == "SYSTOLIC_READING") & (F.col("clinical_value") < 140), 1)
     .when((F.col("eventName") == "DIASTOLIC_READING") & (F.col("clinical_value") < 90), 1)
     .when((F.col("eventName") == "HBA1C_GOOD_CONTROL_INDICATOR") & (F.col("clinical_value") < 7.0), 1)
     .when(F.col("eventName").isin("MAMMOGRAM_DIAGNOSTIC", "HBA1C_LAB_TEST"), 1) # Standard action code closes gap
     .otherwise(0).alias("numerCnt"),

    # --- HEDIS Target & Compliance Dates ---
    F.lit(0.75).alias("expectedRate"), # 75% Performance target benchmark
    
    # Calculating service deadlines based on standard calendar years
    F.to_date(F.concat(F.year(F.col("service_date")), F.lit("-12-31"))).alias("serviceNeededByDate"),
    
    # --- PDC (Proportion of Days Covered) Complex Math Placeholder ---
    # Formula: (Total Days Covered by Prescriptions / Total Days in Target Period)
    F.when(F.col("measureCode") == "SPD-E", 
           F.round((F.col("days_supply") / F.lit(365)) * 100, 2)
    ).otherwise(F.lit(0.0)).alias("PDC")
)